# 01 - Exploratory Data Analysis (EDA)

This notebook focuses on behavior-driven churn insights, not only charts.

## Goal
Understand how activity intensity and recency relate to churn risk, and document actionable findings that can guide feature engineering and retention strategy.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from generate_messy_data import SimulationConfig, simulate_messy_data

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

cfg = SimulationConfig(n_customers=2500, seed=7, reference_date="2024-12-31")
customers_df, transactions_df, logs_df, abt_df = simulate_messy_data(cfg)

print("ABT shape:", abt_df.shape)
abt_df[["churned", "login_count", "total_seconds_active", "days_since_last_login"]].head()

## Activity Intensity Distribution

We first inspect total activity in seconds to understand engagement spread before relating it to churn.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=abt_df, x="total_seconds_active", bins=50, kde=True, color="#1f77b4")
plt.title("Distribution of Total Seconds Active")
plt.xlabel("Total Seconds Active")
plt.ylabel("Customer Count")
plt.tight_layout()
plt.show()

In [ ]:
activity_group = pd.cut(
    abt_df["total_seconds_active"],
    bins=[-1, 300, 500, float("inf")],
    labels=["<300s", "300-500s", ">500s"],
)

churn_by_activity = (
    abt_df.assign(activity_group=activity_group)
    .groupby("activity_group", observed=False)["churned"]
    .mean()
    .rename("churn_rate")
    .reset_index()
)

base_low = float(churn_by_activity.loc[churn_by_activity["activity_group"] == "<300s", "churn_rate"].iloc[0])
base_high = float(churn_by_activity.loc[churn_by_activity["activity_group"] == ">500s", "churn_rate"].iloc[0])
ratio = (base_low / base_high) if base_high > 0 else float("inf")

plt.figure(figsize=(8, 4.5))
sns.barplot(data=churn_by_activity, x="activity_group", y="churn_rate", hue="activity_group", dodge=False, legend=False)
plt.title("Churn Rate by Engagement Band")
plt.xlabel("Engagement band")
plt.ylabel("Churn rate")
plt.tight_layout()
plt.show()

print(churn_by_activity)
print(f"Relative churn risk (<300s vs >500s): {ratio:.2f}x")

### Insight Takeaway

**Observation:** The activity distribution is not uniform; it separates low-engagement and high-engagement customer groups. Customers with very low activity (`< 300s`) show materially higher churn risk than highly engaged users (`> 500s`).

**Business implication:** Early engagement is a leading signal. Product and CRM teams should trigger activation nudges for users whose first weeks show low active time.

**Modeling implication:** `total_seconds_active`, engagement bands, and recency signals should remain high-priority features in the churn model.

## Recency vs Churn Story

A second behavior perspective is recency. Users who have not logged in recently are expected to have elevated churn risk.

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=abt_df, x="churned", y="days_since_last_login")
plt.title("Days Since Last Login by Churn Label")
plt.xlabel("Churned (0 = No, 1 = Yes)")
plt.ylabel("Days Since Last Login")
plt.tight_layout()
plt.show()

### Insight Takeaway

**Observation:** Churned users have significantly higher `days_since_last_login` compared with retained users.

**Business implication:** Login recency is suitable for risk-triggered campaigns (for example, lifecycle reminders after inactivity windows).

**Modeling implication:** Recency features should be kept as first-class predictive signals, and thresholds can be aligned with intervention policies.